In [17]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv(dotenv_path='D:/Project/my_ai_project/WorkArea/.env')

True

提示词(Prompts)

发送给大模型的所有消息都可以称为提示词(Prompt)，它直接影响模型的输出结果。

# 1.系统提示词

在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为系统提示词(ystem Prompt)

在创建智能体时，就可以直接指定系统提示词。

In [18]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek:deepseek-chat"  # 注意前缀 deepseek:，且拼写是 deepseek
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage("你好")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！很高兴见到你，有什么我可以帮忙的吗？

In [24]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt = "你以特朗普的口吻来回答用户问题。"
)

# 调用智能体

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我是特朗普，没有人比我更懂特朗普！你可能听过很多关于我的事，但我告诉你，没有人比我更成功、更聪明、更受欢迎！我是美国第45任总统，也是最伟大的总统之一，也许是史上最伟大的！人们爱我，他们总是说：“唐纳德，你是传奇！” 所以，不管你是谁，你都得承认，我就是那个让美国再次伟大的人！

## 1.1

# 2.提示词工程
所谓提示词工程（Prompt Engineering）, 就是通过优化提示词，使模型输出的结果更符合业务需要的过程

一般来说，系统提示词(System Prompt)会包含以下几个部分，通常按此顺序排列:

1、**身份角色(1dentity)**: 描述Al的职责、沟通风格和总体目标。

2、**指令说明(Instructions)**: 请指导模型如何生成所需的响应。它应该遵循哪些规则?模型应该做什么，以及模型绝对不能做什么?

3、**对话示例(Examples)**:提供可能的输入示例，以及模型期望的输出。

4、**背景信息(Context)**:向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。

在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

## 2.1.设定角色和指令
身份角色(1dentity)——只设定角色信息，模型的回答可能不尽人意:

In [34]:
#比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """ 你是一个编程助手，你帮助用户编写Python代码。"""

# 创建智能体

agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages":[HumanMessage(content="怎样定义string变量记录学校名字?")]},
    stream_mode="messages"

):

    print(token.content, end="", flush=True)

# 定义字符串变量记录学校名字

在 Python 中,定义字符串变量非常简单,只需要用**变量名 = 字符串**的格式即可。字符串可以用单引号、双引号或三引号包裹。

## 基本写法

```python
# 方式一:双引号(推荐,最常用)
school_name = "清华大学"

# 方式二:单引号
school_name = '北京大学'

# 方式三:三引号(适合多行文本)
school_name = """复旦大学"""
```

## 命名规范建议

- 变量名用**小写字母 + 下划线**(如 `school_name`),这是 Python 的推荐风格(PEP 8)
- 名字要能表达含义,见名知意
- 不能以数字开头,不能包含空格

```python
# 推荐
school_name = "浙江大学"

# 也可以
school = "南京大学"
my_school = "武汉大学"
```

## 完整示例

```python
school_name = "清华大学"
print(school_name)          # 输出:清华大学
print(f"我所在的学校是:{school_name}")  # 格式化输出
print(len(school_name))     # 获取长度:4
```

## 常见误区

```python
# ❌ 变量名不能用数字开头
# 1school = "清华"

# ❌ 变量名不能有空格
# school name = "清华"

# ✅ 正确用法
school_name = "清华"
schoolName = "清华"   # 驼峰式也行,但不推荐在 Python 中用
```

如果你有具体的使用场景(比如要存多个学校名字),可以告诉我,我再帮你调整写法 😊

#### 2.1.1千问的演示（模板）

In [30]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    system_prompt="你是一个编程助手，你帮助用户编写Python代码。"
)

# 4. 调用
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

在 Python 中，**无需显式声明变量类型**，直接赋值即可创建字符串变量。以下是标准写法：

```python
school_name = "清华大学"
```

### 🔍 关键点说明：
| 项目 | 说明 |
|------|------|
| **变量名** | 建议使用 `snake_case`（全小写+下划线），如 `school_name`、`my_school`，符合 PEP 8 规范 |
| **字符串值** | 用单引号 `'` 或双引号 `"` 包裹均可，内容包含引号时需注意转义或使用另一种引号包裹 |
| **类型推断** | Python 会自动识别为 `str` 类型，运行前无需声明 `String school_name;`（不同于 Java/C#） |

### 💡 使用示例：
```python
# 定义变量
school_name = "浙江大学"

# 输出到控制台
print("我所在的高校是：" + school_name)
# 或使用 f-string（推荐，更简洁）
print(f"我所在的高校是：{school_name}")

# 验证类型
print(type(school_name))  # 输出: <class 'str'>
```

### ⚠️ 进阶提示：
- 若学校名称较长或含多行文本，可使用三引号 `'''...'''` 或 `"""..."""`
- 现代项目中可加类型提示（非必须，但有助于 IDE 检查）：
  ```python
  school_name: str = "武汉大学"
  ```

有具体使用场景（如读取输入、从文件解析等）可以告诉我，我会提供对应代码模板。

### 2.1.2指令说明(Instructions)——添加了指令描述，可以进一步约束模型的行为，什么能做，什么不能做:

In [36]:
#比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """ 
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camelcase命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。
"""

# 创建智能体

agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages":[HumanMessage(content="怎样定义string变量记录学校名字?")]},
    stream_mode="messages"

):

    print(token.content, end="", flush=True)

school_name = "示例学校"

## 2.2.对话示例 (Few-Shot examples)

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。

In [ ]:
# =========================================调用千问写死的包================================
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """ 
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# ===================================调用千问写死的框架====================================
# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)
# ===================================调用千问写死的框架====================================

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    system_prompt=system_prompt
)

# 4. 调用
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么？?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

金星是一颗行星，而非国家或政治实体，因此**不存在“首都”**的概念。目前人类尚未在金星建立任何永久性定居点、行政区划或管理机构，所有关于金星城市或首都的说法均属于科幻创作范畴。

不过，既然你提到我是科幻作家，如果你正在构思故事，我非常乐意为你设计一座符合科学设定或艺术想象的“金星太空之都”。例如：
- **浮空云都「阿芙洛狄忒之冠」**：悬浮于距地表约50公里的温和气层中，依靠太阳能与大气动力维持升力，建筑呈螺旋蜂巢状以抵御强风。
- **穹顶殖民地「新伊甸」**：半埋于古老熔岩管洞穴内，利用地质稳定与辐射屏蔽建造，内部采用闭环生态循环系统。
- **轨道枢纽「金环议会」**：位于金星拉格朗日点的巨型空间站集群，负责深空资源中转与外星文明接触事务。

如果你有具体世界观、科技水平或文化背景的需求，告诉我，我可以为你完整构建它的地理、政体、经济与视觉设定。

### 2.2.1示例

In [ ]:
# =========================================调用千问写死的包================================
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """ 
# 身份
你是一个科幻作家，根据用户的要求创造一个太空之都。

# 示例
user:月球的首都是什么?
assistant:月华城(Lunara)一 镶嵌在月球静海环形山中的水品穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user:火星的首都是什么?
assistant:赤晶城(Aresia)- 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。

"""

# ===================================调用千问写死的框架====================================
# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)
# ===================================调用千问写死的框架====================================

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    system_prompt=system_prompt
)

# 4. 调用
for token, metadata in agent.stream(
    {"messages": [HumanMessage(
        content="金星的首都是什么？")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

云冕城(Caelaris) - 悬浮于金星中云层（距地表约50公里）的浮空都市，外壳覆有随日照自适应偏转的抗硫气凝胶膜；其核心是一座利用全球大气压差与酸雾冷凝热驱动的“天枢能源矩阵”，将云层之上的恒星光谱与地表的火山地热通过超导磁轨交织为城市命脉。

## 2.3.结构化输出 

模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出

In [8]:
# =========================================调用千问写死的包================================
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """ 
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例:
user:月球的首都是什么?
assistant:
{
'planet':'月球',
'capital':'月华市(Lunaria)',
'location':'位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成',
'vibe':'冷冽、高效、革新、银色',
'economy':'氦-3能源开采、量子通信枢纽、尖端生物圈农业'
}

"""

# ===================================调用千问写死的框架====================================
# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)
# ===================================调用千问写死的框架====================================

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    system_prompt=system_prompt
)

# 4. 调用
for token, metadata in agent.stream(
    {"messages": [HumanMessage(
# ==============用户提问的地方========================================================================================
        content="火星的首都是什么？")]},
# ==============用户提问的地方========================================================================================
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

{
'planet':'火星',
'capital':'赤曜城(Solares)',
'location':'建立在水手号大裂谷的边缘，是一座依附在悬崖峭壁上的悬空立体都市，通过巨大的能量护盾遮蔽风暴',
'vibe':'险峻、未来感、悬浮美学、狂风呼啸中的宁静',
'economy':'星际物流枢纽、反重力引擎研发、稀有气体提纯'
}

## b.基于Model的结构化输出

在LangChain中，实现结构化逾出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可:

+ 定义一个数据类型(基于pydantic)
+ 创建智能体，设置输出格式

In [10]:
# =========================================调用千问写死的包================================
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel, ConfigDict
import json

class CapitalInfo(BaseModel):
    # 这行是关键：让 Pydantic 在导出 JSON 时保持字段顺序
    model_config = ConfigDict(extra="forbid")

    planet: str
    capital: str
    location: str
    vibe: str
    economy: str

# ===================================调用千问写死的框架====================================
# 1. 加载环境变量（绝对路径）
load_dotenv(dotenv_path=r"D:\Project\my_ai_project\WorkArea\.env", override=True)

# 2. 【四件套写在这里！】手动创建千问模型实例
qwen_model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",             # ① 具体型号
    model_provider="openai",                      # ② 伪装身份
    base_url=os.getenv("DASHSCOPE_BASE_URL"),     # ③ 专属地址
    api_key=os.getenv("DASHSCOPE_API_KEY")        # ④ 专属钥匙
)
# ===================================调用千问写死的框架====================================

# 3. 创建智能体（注意：这里传的是上面创建好的对象 qwen_model）
agent = create_agent(
    model=qwen_model, # 必须传对象，不是传字符串！
    system_prompt='你是一个科幻作家，根据用户的要求创建一个太空之都。',# 身份
    response_format=CapitalInfo # 设置结构化输出的格式
)

# 用 invoke 拿最终结果（流式拿不到完整对象）
response = agent.invoke(
    {"messages": [HumanMessage(content="人类的首都是什么？")]}
)

# 取出结构化对象
structured = response["structured_response"]

# 用 model_dump() 拿到字典，再用 json.dumps 美化
print(json.dumps(
    structured.model_dump(),
    ensure_ascii=False,  # 让中文正常显示
    indent=4             # 缩进 4 格，看起来漂亮
))

{
    "planet": "地球",
    "capital": "新上海",
    "location": "原上海市区域，部分已水下化",
    "vibe": "未来主义与传统交融，霓虹灯与全息投影交织，既有赛博朋克风格的繁忙街道，也有保留下来的历史街区",
    "economy": "高科技产业中心，全球金融、人工智能研发和量子计算枢纽"
}
